# Lab 09 — Código de Repetición: Detección y Corrección de Errores

El **código de repetición de 3 qubits** es el QEC más simple: codifica un qubit lógico
en tres qubits físicos y detecta errores de bit-flip (canal X) midiendo síndromes
de paridad sin destruir la información.

**Principio**: |0_L⟩ = |000⟩, |1_L⟩ = |111⟩. Un bit-flip en qubit i produce un
síndrome único que identifica qué qubit falló.

## 1. Codificación del qubit lógico

Para codificar |ψ⟩ = α|0⟩ + β|1⟩:
- α|0⟩ + β|1⟩ → α|000⟩ + β|111⟩

Se logra con CNOT desde el qubit lógico a los dos qubits auxiliares (todos en |0⟩ inicialmente).

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
import numpy as np

def encode_repetition(state='0') -> QuantumCircuit:
    """Codifica |0⟩ o |1⟩ como código de repetición 3-qubit."""
    qc = QuantumCircuit(3)
    if state == '1':
        qc.x(0)  # qubit lógico en |1⟩
    qc.cx(0, 1)  # copia a qubit 1
    qc.cx(0, 2)  # copia a qubit 2
    return qc

# Verificar codificación
qc0 = encode_repetition('0')
qc1 = encode_repetition('1')
sv0 = Statevector(qc0)
sv1 = Statevector(qc1)
print('Codificación |0_L⟩ = |000⟩:')
print(f'  Probs: {sv0.probabilities_dict()}')
print('Codificación |1_L⟩ = |111⟩:')
print(f'  Probs: {sv1.probabilities_dict()}')

## 2. Introducir un error de bit-flip

Simulamos un error de canal X en uno de los tres qubits físicos.
El código debe detectar en cuál ocurrió sin medir directamente los datos.

In [ ]:
def introduce_error(qc: QuantumCircuit, qubit: int) -> QuantumCircuit:
    """Aplica un bit-flip X en el qubit indicado."""
    qc_err = qc.copy()
    qc_err.x(qubit)
    return qc_err

# Mostrar el efecto del error en cada qubit
for err_qubit in [0, 1, 2, None]:
    qc = encode_repetition('0')
    if err_qubit is not None:
        qc = introduce_error(qc, err_qubit)
    sv = Statevector(qc)
    top = max(sv.probabilities_dict(), key=lambda k: sv.probabilities_dict()[k])
    label = f'Error en qubit {err_qubit}' if err_qubit is not None else 'Sin error'
    print(f'{label:<22}: estado dominante = |{top}⟩')

## 3. Circuito de síndrome

El circuito de síndrome usa 2 qubits ancilla para medir paridades sin colapsar el estado lógico:
- **Síndrome 0** (s0): paridad entre qubits 0 y 1 → s0 = Z₀ ⊕ Z₁
- **Síndrome 1** (s1): paridad entre qubits 1 y 2 → s1 = Z₁ ⊕ Z₂

| s0 | s1 | Error |
|----|----|---------|
| 0  | 0  | Sin error |
| 1  | 0  | Qubit 0 |
| 1  | 1  | Qubit 1 |
| 0  | 1  | Qubit 2 |

In [ ]:
def syndrome_circuit(qc_data: QuantumCircuit) -> QuantumCircuit:
    """Añade medición de síndrome a un circuito de 3 qubits de datos."""
    qr_synd = QuantumRegister(2, 'synd')
    cr_synd = ClassicalRegister(2, 's')
    qc = QuantumCircuit(qc_data.qregs[0], qr_synd, cr_synd)
    qc.compose(qc_data, qubits=range(3), inplace=True)

    # Síndrome 0: paridad qubits 0,1
    qc.cx(0, 3)
    qc.cx(1, 3)
    # Síndrome 1: paridad qubits 1,2
    qc.cx(1, 4)
    qc.cx(2, 4)
    qc.measure([3, 4], [0, 1])
    return qc

sampler = StatevectorSampler()
print('Detección de errores:')
print(f'{"Escenario":<25} | Síndrome | Error detectado')
print('-' * 60)
syndrome_table = {'00': 'Sin error', '10': 'Qubit 0', '11': 'Qubit 1', '01': 'Qubit 2'}

for err_q, label in [(None,'Sin error'), (0,'Error qubit 0'), (1,'Error qubit 1'), (2,'Error qubit 2')]:
    qc_base = encode_repetition('0')
    if err_q is not None:
        qc_base = introduce_error(qc_base, err_q)
    qr_data = QuantumRegister(3, 'data')
    qc_base2 = QuantumCircuit(qr_data)
    qc_base2.compose(qc_base, inplace=True)
    qc_s = syndrome_circuit(qc_base2)
    job = sampler.run([qc_s], shots=256)
    counts = job.result()[0].data.s.get_counts()
    synd = max(counts, key=counts.get)
    print(f'{label:<25} | s={synd}    | {syndrome_table.get(synd, "?")}')

## 4. Corrección automática

Con el síndrome, aplicamos la corrección X en el qubit indicado.
Verificamos que el estado lógico se recupera con fidelidad 1.

In [ ]:
from qiskit.quantum_info import state_fidelity

def correct_and_decode(qc_data: QuantumCircuit, logical: str) -> float:
    """Mide síndrome clásicamente y aplica corrección. Retorna fidelidad final."""
    # En simulación obtenemos el síndrome determinístico
    qr_d = QuantumRegister(3, 'data')
    qr_s = QuantumRegister(2, 'synd')
    cr_s = ClassicalRegister(2, 's')
    qc = QuantumCircuit(qr_d, qr_s, cr_s)
    qc.compose(qc_data, qubits=range(3), inplace=True)
    qc.cx(0, 3); qc.cx(1, 3)  # synd 0
    qc.cx(1, 4); qc.cx(2, 4)  # synd 1

    # Obtener síndrome desde el statevector
    sv_pre = Statevector(qc)
    probs = sv_pre.probabilities_dict()
    top_state = max(probs, key=probs.get)
    # síndrome = bits ancilla (qubits 3,4)
    synd = top_state[-2:]  # últimos 2 bits = ancilla

    # Corrección
    corrections = {'00': None, '10': 0, '11': 1, '01': 2}
    fix = corrections.get(synd)

    qc_corrected = qc_data.copy()
    if fix is not None:
        qc_corrected.x(fix)

    # Decodificar (medir qubit 0)
    sv_final = Statevector(qc_corrected)
    from qiskit.quantum_info import partial_trace, DensityMatrix
    dm = DensityMatrix(sv_final)
    dm_q0 = partial_trace(dm, [1, 2])
    ideal = DensityMatrix.from_label(logical)
    return float(state_fidelity(dm_q0, ideal))

print('Corrección QEC — fidelidad tras corrección:')
for logical in ['0', '1']:
    for err_q in [None, 0, 1, 2]:
        qc_base = QuantumCircuit(QuantumRegister(3, 'data'))
        if logical == '1': qc_base.x(0)
        qc_base.cx(0, 1); qc_base.cx(0, 2)
        if err_q is not None: qc_base.x(err_q)
        fid = correct_and_decode(qc_base, logical)
        err_label = f'err q{err_q}' if err_q is not None else 'sin err'
        print(f'  |{logical}_L⟩, {err_label}: F = {fid:.4f}')

## 5. Limitaciones: el código de repetición solo corrige X

El código de repetición clásico falla ante errores de fase (Z). Un error Z en cualquier qubit no cambia la paridad ⟹ síndrome = 00 ⟹ no detectado.

Para proteger contra X **y** Z simultáneamente se necesitan códigos más potentes
(Shor 9-qubit, Steane 7-qubit, surface codes).

| Código | Qubits físicos | Protege contra | Distancia |
|--------|---------------|----------------|----------|
| Repetición 3q | 3 | X solamente | 3 |
| Shor 9q | 9 | X y Z | 3 |
| Steane 7q | 7 | X y Z | 3 |
| Surface d=3 | 9 datos + 8 ancilla | X y Z | 3 |

In [ ]:
# Demostración: error Z no detectado
qc_base = QuantumCircuit(QuantumRegister(3, 'data'))
qc_base.cx(0, 1); qc_base.cx(0, 2)  # codifica |0_L⟩
qc_base.z(1)  # error de fase en qubit 1

qr_d = QuantumRegister(3, 'data')
qr_s = QuantumRegister(2, 'synd')
cr_s = ClassicalRegister(2, 's')
qc = QuantumCircuit(qr_d, qr_s, cr_s)
qc.compose(qc_base, qubits=range(3), inplace=True)
qc.cx(0, 3); qc.cx(1, 3)
qc.cx(1, 4); qc.cx(2, 4)
qc.measure([3, 4], [0, 1])

job = sampler.run([qc], shots=256)
counts = job.result()[0].data.s.get_counts()
print('Error Z en qubit 1:')
print(f'  Síndrome medido: {max(counts, key=counts.get)}')
print('  → Síndrome 00 = no detectado (el código de repetición es ciego a errores Z)')